# Чекпойнт 2. Разведочный анализ данных (EDA): фьючерс Solana (SOLUSDT perpetual)

**Цель этапа:** изучить структуру и качество исторических данных бессрочного фьючерса SOLUSDT, свойства его доходности и волатильности, связь с Bitcoin, а также данные, которые есть только у фьючерсов (базис к споту и ставку финансирования), и сформулировать выводы для этапа моделирования.

**Почему фьючерсы.** Итоговая торговая стратегия будет работать на бессрочных фьючерсах: на них можно открывать и длинные, и короткие позиции, поэтому прогноз падения тоже можно использовать. Кроме того, фьючерсы дают дополнительные признаки, которых нет у спота: ставку финансирования (funding rate) и базис.

**Данные:** дневные свечи бессрочных фьючерсов SOLUSDT и BTCUSDT (Binance USDⓈ-M), история funding rate по обоим контрактам, дневные спотовые свечи SOL/USDT и BTC/USDT для сравнения. Загрузка: `python src/load_data.py`; запасной источник фьючерсов — Bybit, спота — Yahoo Finance.

**Содержание**
1. Качество данных
2. Цена и объём
3. Доходности и их распределение
4. Стационарность
5. Автокорреляция и кластеризация волатильности
6. Волатильность
7. Взаимосвязь SOL и BTC
8. Фьючерс и спот: базис
9. Ставка финансирования (funding rate)
10. Анализ по подпериодам (проверка устойчивости)
11. Итоги и выводы для моделирования

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

INTERVAL = "1d"     # должен совпадать с --interval у src/load_data.py
MARKET = "futures"  # основной ряд: бессрочный фьючерс; "spot" — чтобы повторить анализ на споте
PERIODS_PER_YEAR = {"1d": 365, "4h": 365 * 6, "1h": 365 * 24}[INTERVAL]
ROLL = {"1d": 30, "4h": 180, "1h": 720}[INTERVAL]   # окно ≈ 30 дней
FREQ = {"1d": "D", "4h": "4h", "1h": "h"}[INTERVAL]

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW, PROCESSED = ROOT / "data" / "raw", ROOT / "data" / "processed"
COLORS = {"sol": "#7B3FE4", "btc": "#E8890C"}


def load(asset: str, market: str = MARKET) -> pd.DataFrame:
    df = pd.read_csv(RAW / f"{asset}_{market}_{INTERVAL}.csv", parse_dates=["timestamp"])
    return df.set_index("timestamp").sort_index()


def load_funding(asset: str) -> pd.Series:
    df = pd.read_csv(RAW / f"{asset}_funding.csv", parse_dates=["timestamp"])
    return df.set_index("timestamp").sort_index()["funding_rate"]


def usd_volume(df: pd.DataFrame) -> pd.Series:
    # Объём в долларах: quote_volume у Binance (turnover у Bybit); у Yahoo столбец volume уже в долларах.
    return df["quote_volume"] if "quote_volume" in df.columns else df["volume"]


sol, btc = load("sol"), load("btc")                         # фьючерсы — основной ряд
sol_spot, btc_spot = load("sol", "spot"), load("btc", "spot")  # спот — для сравнения (раздел 8)
fund_sol, fund_btc = load_funding("sol"), load_funding("btc")  # funding rate (раздел 9)
print(f"Фьючерс SOLUSDT: {sol.index.min():%Y-%m-%d} — {sol.index.max():%Y-%m-%d}, {len(sol)} свечей")
sol.tail()

## 1. Качество данных

Проверяем период, дыры во временном ряду (пропущенные свечи), дубликаты, пропуски, нулевой объём и логическую согласованность OHLC (`high` не ниже `open`/`close`, `low` не выше).

In [ ]:
def quality_report(df: pd.DataFrame, name: str) -> pd.Series:
    full_index = pd.date_range(df.index.min(), df.index.max(), freq=FREQ)
    body_max = df[["open", "close"]].max(axis=1)
    body_min = df[["open", "close"]].min(axis=1)
    return pd.Series({
        "начало": df.index.min().date(),
        "конец": df.index.max().date(),
        "свечей": len(df),
        "пропущенных свечей": len(full_index.difference(df.index)),
        "дубликатов по времени": int(df.index.duplicated().sum()),
        "пропусков (NaN)": int(df.isna().sum().sum()),
        "свечей с нулевым объёмом": int((df["volume"] == 0).sum()),
        "нарушений high/low": int(((df["high"] < body_max) | (df["low"] > body_min)).sum()),
        "неположительных цен": int((df[["open", "high", "low", "close"]] <= 0).sum().sum()),
    }, name=name)


pd.concat([quality_report(sol, "SOL"), quality_report(btc, "BTC")], axis=1)

In [ ]:
sol[["open", "high", "low", "close", "volume"]].describe().T

**Вывод.** Данные полные и чистые: за период 11.08.2020 — 28.10.2026 по каждой паре 2270 дневных свечей, нет пропущенных дней, дубликатов, NaN, дней с нулевым объёмом и логических ошибок OHLC. Дополнительная очистка не нужна.

## 2. Цена и объём

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(sol["close"], color=COLORS["sol"])
axes[0].set_yscale("log")
axes[0].set_title("SOL/USDT, цена закрытия (логарифмическая шкала)")

axes[1].plot(btc["close"], color=COLORS["btc"])
axes[1].set_yscale("log")
axes[1].set_title("BTC/USDT, цена закрытия (логарифмическая шкала)")

axes[2].plot(usd_volume(sol).rolling(ROLL).mean() / 1e6, color=COLORS["sol"])
axes[2].set_title(f"Объём торгов SOL, млн $ (скользящее среднее, окно {ROLL})")

plt.tight_layout()
plt.show()

In [ ]:
# Рост 1 $, вложенного в начале периода
growth = pd.DataFrame({"SOL": sol["close"] / sol["close"].iloc[0],
                       "BTC": btc["close"] / btc["close"].iloc[0]})
ax = growth.plot(logy=True, color=[COLORS["sol"], COLORS["btc"]],
                 title="Рост 1 $ с начала периода (логарифмическая шкала)")
ax.set_xlabel("")
plt.show()

drawdown = growth / growth.cummax() - 1
print("Максимальная просадка:", drawdown.min().map("{:.1%}".format).to_dict())

**Вывод.** На графиках чётко видны фазы рынка: взрывной рост SOL в 2021 году (примерно с $3 до $260), обвал 2022 года (крах Terra/LUNA в мае и биржи FTX в ноябре), восстановление в 2023–2024 годах. Максимальная просадка от пика составила **−96.3% у SOL** и −76.6% у BTC: SOL — существенно более рискованный актив. Цена менялась почти на два порядка, поэтому дальше работаем с логарифмами и доходностями, а не с ценой.

## 3. Доходности и их распределение

Используем логарифмическую доходность $r_t = \ln(P_t / P_{t-1})$: она аддитивна во времени (доходность за $h$ дней равна сумме дневных) и симметричнее простой доходности.

In [ ]:
ret = pd.DataFrame({"sol": np.log(sol["close"]).diff(),
                    "btc": np.log(btc["close"]).diff()}).dropna()


def describe_returns(r: pd.Series) -> pd.Series:
    sigma = r.std()
    return pd.Series({
        "среднее": r.mean(),
        "медиана": r.median(),
        "ст. отклонение": sigma,
        "годовая волатильность": sigma * np.sqrt(PERIODS_PER_YEAR),
        "асимметрия": stats.skew(r),
        "эксцесс (excess)": stats.kurtosis(r),
        "минимум": r.min(),
        "максимум": r.max(),
        "доля |r| > 3σ": (r.abs() > 3 * sigma).mean(),
        "то же у нормального": 2 * stats.norm.sf(3),
        "доля положительных": (r > 0).mean(),
        "Жарк–Бера, p-value": stats.jarque_bera(r).pvalue,
    })


ret_stats = ret.apply(describe_returns).rename(columns=str.upper)
ret_stats

In [ ]:
# Самые сильные движения SOL — полезно сопоставить с новостями
top = ret["sol"].abs().nlargest(10).index
pd.DataFrame({"SOL": ret.loc[top, "sol"], "BTC": ret.loc[top, "btc"]}).sort_index()

In [ ]:
r = ret["sol"]
x = np.linspace(r.min(), r.max(), 400)
normal_pdf = stats.norm.pdf(x, r.mean(), r.std())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, log_scale in zip(axes[:2], [False, True]):
    ax.hist(r, bins=120, density=True, color=COLORS["sol"], alpha=0.7, label="SOL")
    ax.plot(x, normal_pdf, "k--", lw=1.2, label="нормальное распределение")
    ax.legend()
    if log_scale:
        ax.set_yscale("log")
        ax.set_ylim(bottom=1e-2)
axes[0].set_title("Распределение дневных лог-доходностей SOL")
axes[1].set_title("То же, логарифмическая шкала: «тяжёлые хвосты»")

stats.probplot(r, dist="norm", plot=axes[2])
axes[2].get_lines()[0].set(color=COLORS["sol"], markersize=3)
axes[2].set_title("QQ-plot относительно нормального")
axes[2].set_xlabel("теоретические квантили")
axes[2].set_ylabel("наблюдаемые доходности")
plt.tight_layout()
plt.show()

**Вывод.**
- Годовая волатильность SOL — **113%**, почти вдвое выше, чем у BTC (57%). Дневное стандартное отклонение ≈ 5.9% против 3.0%.
- Распределение далеко от нормального: **эксцесс 8.5** у SOL (4.0 у BTC), тест Жарка–Бера уверенно отвергает нормальность (p ≈ 0). Движения больше 3σ у SOL случаются в 1.4% дней, это **в ~5 раз чаще**, чем у нормального распределения (0.27%).
- Асимметрия слабо отрицательная (−0.21): сильнейшие падения глубже сильнейших взлётов (минимум −55%, максимум +38% за день).
- Средняя дневная доходность (0.15%) в ~40 раз меньше стандартного отклонения, доля растущих дней почти ровно 50%. Угадать знак завтрашней доходности по одной лишь статистике нельзя.
- Крупнейшие движения SOL совпадают с известными событиями: 19.05.2021 (обвал рынка после запрета финансовым организациям Китая работать с криптовалютой, −47%), 21.06.2021 (запрет майнинга в Китае, −28%), 11.05.2022 (крах Terra/LUNA, −27%), **09.11.2022 (крах FTX, −55%)**, 14.01.2023 (отскок после FTX, +28%). Часть экстремальных дней 2020–2021 годов (например, +38% 09.09.2020) случилась при почти неподвижном BTC: это собственные события SOL на фоне низкой ликвидности после листинга.

**Следствие для модели:** MSE будет перекошена редкими экстремальными днями, поэтому вместе с ней смотрим MAE / Huber loss и точность направления. Стоит проверить устойчивость результатов к исключению 2020 года (период низкой ликвидности).

## 4. Стационарность

Тест Дики–Фуллера (ADF): нулевая гипотеза — у ряда есть единичный корень, то есть ряд нестационарен. Если p-value < 0.05, гипотезу отвергаем.

In [ ]:
series = {
    "log-цена SOL": np.log(sol["close"]),
    "log-цена BTC": np.log(btc["close"]),
    "доходность SOL": ret["sol"],
    "доходность BTC": ret["btc"],
}
rows = {}
for name, s in series.items():
    adf_stat, p_value, *_ = adfuller(s.dropna(), autolag="AIC")
    rows[name] = {"ADF-статистика": adf_stat, "p-value": p_value, "стационарен": p_value < 0.05}
adf_table = pd.DataFrame(rows).T
adf_table

**Вывод.** Лог-цена нестационарна: p-value 0.15 у SOL и 0.25 у BTC, гипотезу о единичном корне отвергнуть нельзя. Доходность стационарна: ADF-статистика ≈ −50, p-value ≈ 0. Это формальное обоснование выбора целевой переменной: **прогнозируем доходность, а не цену**. Модель, обученная на цене, легко покажет «хорошее» качество, просто повторяя вчерашнее значение.

## 5. Автокорреляция и кластеризация волатильности

Смотрим автокорреляцию (ACF) самих доходностей и их квадратов. Пунктир — граница значимости ±1.96/√n. Тест Льюнга–Бокса проверяет, что автокорреляции до лага $h$ одновременно равны нулю.

In [ ]:
NLAGS = 30


def acf(x, nlags: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.dot(x, x)
    return np.array([np.dot(x[k:], x[:len(x) - k]) / denom for k in range(1, nlags + 1)])


def ljung_box_pvalue(x, h: int) -> float:
    n = len(x)
    rho = acf(x, h)
    q = n * (n + 2) * np.sum(rho ** 2 / (n - np.arange(1, h + 1)))
    return stats.chi2.sf(q, h)


fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for row, asset in enumerate(["sol", "btc"]):
    for col, (label, data) in enumerate([("доходности", ret[asset]), ("квадраты доходностей", ret[asset] ** 2)]):
        ax = axes[row, col]
        band = 1.96 / np.sqrt(len(data))
        ax.bar(range(1, NLAGS + 1), acf(data, NLAGS), color=COLORS[asset], width=0.6)
        ax.axhline(band, color="k", ls="--", lw=0.8)
        ax.axhline(-band, color="k", ls="--", lw=0.8)
        ax.set_title(f"ACF: {asset.upper()}, {label}")
axes[1, 0].set_xlabel("лаг")
axes[1, 1].set_xlabel("лаг")
plt.tight_layout()
plt.show()

lb = pd.DataFrame({
    f"{asset.upper()}: {label}": {f"лаг {h}": ljung_box_pvalue(data, h) for h in (5, 10, 20)}
    for asset in ["sol", "btc"]
    for label, data in [("r", ret[asset]), ("r²", ret[asset] ** 2)]
})
print("Льюнг–Бокс, p-value:")
lb

In [ ]:
# PACF (частичная автокорреляция): прямая связь с лагом k после учёта всех промежуточных лагов.
# Считаем рекурсией Дурбина–Левинсона из тех же коэффициентов ACF.
def pacf(x, nlags: int) -> np.ndarray:
    rho = np.r_[1.0, acf(x, nlags)]
    phi = np.zeros((nlags + 1, nlags + 1))
    out = []
    for k in range(1, nlags + 1):
        if k == 1:
            phi[1, 1] = rho[1]
        else:
            num = rho[k] - np.dot(phi[k - 1, 1:k], rho[1:k][::-1])
            den = 1 - np.dot(phi[k - 1, 1:k], rho[1:k])
            phi[k, k] = num / den
            phi[k, 1:k] = phi[k - 1, 1:k] - phi[k, k] * phi[k - 1, 1:k][::-1]
        out.append(phi[k, k])
    return np.array(out)


fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
for ax, asset in zip(axes, ["sol", "btc"]):
    values = pacf(ret[asset], NLAGS)
    band = 1.96 / np.sqrt(len(ret))
    ax.bar(range(1, NLAGS + 1), values, color=COLORS[asset], width=0.6)
    ax.axhline(band, color="k", ls="--", lw=0.8)
    ax.axhline(-band, color="k", ls="--", lw=0.8)
    ax.set_title(f"PACF: {asset.upper()}, доходности")
    ax.set_xlabel("лаг")
plt.tight_layout()
plt.show()

pacf_sol = pacf(ret["sol"], NLAGS)
significant = [k + 1 for k, v in enumerate(pacf_sol) if abs(v) > band]
print(f"PACF SOL, лаги 1–5: {np.round(pacf_sol[:5], 3).tolist()}")
print(f"Значимые лаги PACF SOL (|v| > {band:.3f}): {significant}")

**Вывод.**
- **Квадраты доходностей** сильно автокоррелированы у обеих монет (Льюнг–Бокс, p ≈ 0 на всех лагах): это **кластеризация волатильности**, бурные дни идут сериями.
- **Сами доходности BTC** автокорреляции не имеют (p = 0.16–0.34), то есть по прошлому линейно не предсказываются.
- **У SOL** автокорреляция доходностей слабая, но статистически значимая на длинных лагах (p = 0.012 для 10 лагов, 0.004 для 20). Автокорреляция на лаге 1 равна −0.046, что чуть выходит за границу ±0.041: это намёк на слабый откат после сильных дней. Эффект маленький, но означает, что SOL предсказуема по своему прошлому чуть лучше, чем BTC.
- **PACF** подтверждает картину: у SOL значим только лаг 1 (−0.046) и отдельные дальние лаги (15 и 25). Дальние лаги почти наверняка случайны: при 30 проверках 1–2 ложных срабатывания на уровне 5% ожидаемы. Значит, из прошлых доходностей в признаки достаточно взять несколько ближайших лагов.

**Следствие для модели:** волатильность прогнозируется заметно лучше направления, поэтому признаки волатильности (скользящее стандартное отклонение, размах high–low, ATR) обязательны. Лаги собственной доходности SOL стоит включить в признаки, но ожидать от них многого не стоит. Модели сравниваем с бейзлайнами «нулевая доходность» и «завтра как сегодня».

## 6. Волатильность

In [ ]:
roll_vol = ret.rolling(ROLL).std() * np.sqrt(PERIODS_PER_YEAR)
ax = roll_vol.plot(color=[COLORS["sol"], COLORS["btc"]],
                   title=f"Скользящая годовая волатильность (окно {ROLL})")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(["SOL", "BTC"])
ax.set_xlabel("")
plt.show()

yearly_vol = ret.groupby(ret.index.year).agg(lambda r: r.std() * np.sqrt(PERIODS_PER_YEAR))
yearly_vol.columns = ["SOL", "BTC"]
yearly_vol["SOL / BTC"] = yearly_vol["SOL"] / yearly_vol["BTC"]
yearly_vol.index.name = "год"
yearly_vol

In [ ]:
WEEKDAYS = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
by_weekday = ret.groupby(ret.index.day_of_week).agg(["mean", "std"])
by_weekday.index = WEEKDAYS

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
w = 0.38
xs = np.arange(7)
for i, asset in enumerate(["sol", "btc"]):
    axes[0].bar(xs + (i - 0.5) * w, by_weekday[(asset, "mean")], w, color=COLORS[asset], label=asset.upper())
    axes[1].bar(xs + (i - 0.5) * w, by_weekday[(asset, "std")], w, color=COLORS[asset], label=asset.upper())
for ax, title in zip(axes, ["Средняя доходность по дням недели", "Волатильность по дням недели"]):
    ax.set_xticks(xs, WEEKDAYS)
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

if INTERVAL != "1d":
    by_hour = ret.groupby(ret.index.hour).std()
    by_hour.plot(kind="bar", color=[COLORS["sol"], COLORS["btc"]],
                 title="Волатильность по часам (UTC)", figsize=(14, 4))
    plt.show()

In [ ]:
# Эффект выходных: сравниваем разброс (волатильность) и среднюю доходность
weekend = ret.index.day_of_week >= 5
rows = {}
for asset in ["sol", "btc"]:
    r = ret[asset]
    rows[asset.upper()] = {
        "волатильность в будни": r[~weekend].std(),
        "волатильность в выходные": r[weekend].std(),
        "выходные / будни": r[weekend].std() / r[~weekend].std(),
        "тест Левена (разброс), p-value": stats.levene(r[weekend], r[~weekend]).pvalue,
        "тест Краскела–Уоллиса (среднее по дням), p-value":
            stats.kruskal(*[r[ret.index.day_of_week == d] for d in range(7)]).pvalue,
    }
pd.DataFrame(rows)

Тест Левена проверяет, одинаков ли разброс в двух группах (p < 0.05 — разброс различается). Тест Краскела–Уоллиса проверяет, одинаковы ли распределения доходности по семи дням недели; он не требует нормальности, что важно при тяжёлых хвостах.

**Вывод.**
- Волатильность SOL **снижается год от года**: 169% в 2020, 161% в 2021, 123% в 2022, 96% в 2023, 82% в 2024, 86% в 2025, 62% в 2026. Монета «взрослеет»: растут капитализация и ликвидность.
- Вместе с этим сокращается разрыв с BTC: в 2020 году SOL была в 3 раза волатильнее BTC, в 2026 — лишь в 1.4 раза.
- Есть **эффект выходных**: в субботу и воскресенье волатильность заметно ниже (у SOL 5.0% против 6.2% в будни, у BTC 2.0% против 3.3%; тест Левена, p < 10⁻⁶). При этом средняя доходность по дням недели статистически не различается (тест Краскела–Уоллиса, p = 0.31 у SOL и 0.43 у BTC).

**Следствие для модели:** признак «выходной / будний день» полезен для прогноза волатильности и размера движения, но не направления. Из-за смены режимов модель нужно проверять на нескольких периодах (walk-forward), а признаки волатильности лучше нормировать на текущий уровень.

## 7. Взаимосвязь SOL и BTC

Считаем корреляцию доходностей, бету SOL к BTC ($\beta = \mathrm{cov}(r_{SOL}, r_{BTC}) / \mathrm{var}(r_{BTC})$, на сколько процентов в среднем двигается SOL при движении BTC на 1%), их изменение во времени и проверяем, опережает ли BTC движение SOL.

In [ ]:
pearson = ret["sol"].corr(ret["btc"])
spearman = ret["sol"].corr(ret["btc"], method="spearman")
beta = ret["sol"].cov(ret["btc"]) / ret["btc"].var()
alpha = ret["sol"].mean() - beta * ret["btc"].mean()

print(f"Корреляция Пирсона:   {pearson:.3f}")
print(f"Корреляция Спирмена:  {spearman:.3f}")
print(f"Бета SOL к BTC:       {beta:.2f}")
print(f"Доля дисперсии SOL, объяснённая BTC (R²): {pearson ** 2:.1%}")

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(ret["btc"], ret["sol"], s=6, alpha=0.4, color=COLORS["sol"])
xx = np.linspace(ret["btc"].min(), ret["btc"].max(), 100)
ax.plot(xx, alpha + beta * xx, "k-", lw=1.2, label=f"β = {beta:.2f}")
ax.set_xlabel("доходность BTC")
ax.set_ylabel("доходность SOL")
ax.set_title("Дневные доходности SOL и BTC")
ax.legend()
plt.show()

In [ ]:
W = 3 * ROLL  # окно ≈ 90 дней
roll_corr = ret["sol"].rolling(W).corr(ret["btc"])
roll_beta = ret["sol"].rolling(W).cov(ret["btc"]) / ret["btc"].rolling(W).var()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(roll_corr, color=COLORS["sol"])
axes[0].axhline(pearson, color="k", ls="--", lw=0.8, label="за весь период")
axes[0].set_title(f"Скользящая корреляция SOL–BTC (окно {W})")
axes[0].legend()
axes[1].plot(roll_beta, color=COLORS["sol"])
axes[1].axhline(beta, color="k", ls="--", lw=0.8, label="за весь период")
axes[1].set_title(f"Скользящая бета SOL к BTC (окно {W})")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Как связь с BTC менялась во времени: корреляция и бета по календарным годам
by_year = ret.groupby(ret.index.year)
yearly_link = pd.DataFrame({
    "корреляция": by_year.apply(lambda d: d["sol"].corr(d["btc"])),
    "бета": by_year.apply(lambda d: d["sol"].cov(d["btc"]) / d["btc"].var()),
    "дней": by_year.size(),
})
yearly_link.index.name = "год"

print(f"Скользящая корреляция ({W} дней): от {roll_corr.min():.2f} ({roll_corr.idxmin():%Y-%m-%d}) "
      f"до {roll_corr.max():.2f} ({roll_corr.idxmax():%Y-%m-%d})")
print(f"Скользящая бета ({W} дней):       от {roll_beta.min():.2f} ({roll_beta.idxmin():%Y-%m-%d}) "
      f"до {roll_beta.max():.2f} ({roll_beta.idxmax():%Y-%m-%d})")
yearly_link

In [ ]:
# Кросс-корреляция: corr(SOL_t, BTC_{t-k}).
# k > 0: BTC опережает SOL на k периодов (это и есть потенциально полезный признак),
# k < 0: SOL опережает BTC.
LAGS = range(-5, 6)
xcorr = pd.Series({k: ret["sol"].corr(ret["btc"].shift(k)) for k in LAGS})
band = 1.96 / np.sqrt(len(ret))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(xcorr.index, xcorr.values, color=[COLORS["btc"] if k == 0 else COLORS["sol"] for k in xcorr.index])
ax.axhline(band, color="k", ls="--", lw=0.8)
ax.axhline(-band, color="k", ls="--", lw=0.8)
ax.set_xticks(list(LAGS))
ax.set_xlabel("лаг k (k > 0: BTC опережает SOL)")
ax.set_title("Кросс-корреляция доходностей SOL и BTC")
plt.show()

lead = pd.DataFrame({
    "corr(SOL_{t+k}, BTC_t)": {k: ret["sol"].shift(-k).corr(ret["btc"]) for k in range(1, 6)},
    "corr(SOL_{t+k}, SOL_t)": {k: ret["sol"].shift(-k).corr(ret["sol"]) for k in range(1, 6)},
})
lead.index.name = "горизонт k"
print(f"Граница значимости ±{band:.3f}")
lead

**Вывод.**
- Связь SOL с BTC существенная, но далеко не полная: корреляция Пирсона **0.59** (Спирмена 0.64), BTC объясняет **~35%** дисперсии SOL, остальные 65% — собственные движения SOL.
- **Бета 1.17**: при движении BTC на 1% SOL в среднем двигается на 1.17% в ту же сторону.
- Связь **сильно усиливается со временем**: корреляция по годам 0.29 (2020) → 0.42 (2021) → 0.79 (2022) → 0.62 (2023) → 0.75 (2024) → 0.80 (2025) → 0.88 (2026). Скользящая 90-дневная корреляция менялась от 0.08 (май 2021) до 0.93 (апрель 2026), бета — от 0.18 до 2.77. Ранняя SOL жила своей жизнью, сейчас она движется почти синхронно с рынком.
- **Опережения почти нет**: вне нулевого лага все кросс-корреляции малы (|corr| ≤ 0.064). Вчерашний BTC не предсказывает завтрашний SOL: corr(SOL_t+1, BTC_t) = −0.035, это внутри границы ±0.041. Пограничные значения есть на лаге 2 (0.055) и лаге −1 (−0.064), но они маленькие и могут быть случайными при множественном сравнении.

**Следствие для модели:** признаки BTC полезны в первую очередь как описание рыночного режима (волатильность и тренд BTC, скользящая корреляция и бета), а не как прямой предиктор. Поскольку связь со временем меняется, скользящие корреляцию и бету стоит включить как признаки. Опережение BTC стоит перепроверить на часовых данных.

## 8. Фьючерс и спот: базис

**Базис** — относительная разница между ценой фьючерса и спота: $basis_t = F_t / S_t - 1$. Бессрочный фьючерс удерживается около спота механизмом funding: если фьючерс дороже спота, длинные позиции платят коротким, и наоборот. Проверяем, насколько точно фьючерс повторяет спот (можно ли переносить выводы) и сколько информации несёт сам базис.

In [ ]:
basis = pd.DataFrame({
    asset.upper(): (fut["close"] / spot["close"]).dropna() - 1
    for asset, fut, spot in [("sol", sol, sol_spot), ("btc", btc, btc_spot)]
}).dropna()

ret_spot = pd.DataFrame({"sol": np.log(sol_spot["close"]).diff(),
                         "btc": np.log(btc_spot["close"]).diff()})
common = ret.index.intersection(ret_spot.dropna().index)
tracking = pd.DataFrame({
    a.upper(): {
        "корреляция доходностей фьючерс–спот": ret.loc[common, a].corr(ret_spot.loc[common, a]),
        "ст. отклонение разницы доходностей": (ret.loc[common, a] - ret_spot.loc[common, a]).std(),
        "средний базис": basis[a.upper()].mean(),
        "медианный базис": basis[a.upper()].median(),
        "мин. базис": basis[a.upper()].min(),
        "макс. базис": basis[a.upper()].max(),
        "доля дней с базисом > 0 (контанго)": (basis[a.upper()] > 0).mean(),
    } for a in ["sol", "btc"]
})

fig, ax = plt.subplots(figsize=(12, 4))
for a in ["sol", "btc"]:
    ax.plot(basis[a.upper()].rolling(7).mean() * 100, color=COLORS[a], label=a.upper(), lw=1)
ax.axhline(0, color="k", lw=0.8)
ax.set_title("Базис фьючерс − спот, % (скользящее среднее за 7 дней)")
ax.legend()
plt.show()
tracking

_Вывод будет дописан по результатам запуска на реальных данных._

## 9. Ставка финансирования (funding rate)

**Funding rate** — периодический платёж между длинными и короткими позициями бессрочного фьючерса (на Binance обычно раз в 8 часов). Положительная ставка: лонги платят шортам — рынок перегрет покупателями. Отрицательная: шорты платят лонгам — преобладают продавцы. Это показатель настроения толпы с плечом, которого нет у спотовых цен.

Для анализа суммируем платежи за сутки (дневной funding) и смотрим: какой он обычно, как меняется, связан ли с будущей доходностью и во сколько обходится удержание позиции.

In [ ]:
fund_daily = pd.DataFrame({"sol": fund_sol.resample("D").sum(min_count=1),
                           "btc": fund_btc.resample("D").sum(min_count=1)}).dropna()
fund_daily.index = fund_daily.index.normalize()

fund_table = pd.DataFrame({
    a.upper(): {
        "записей": len(f),
        "период": f"{f.index.min():%Y-%m-%d} — {f.index.max():%Y-%m-%d}",
        "типичный интервал, ч": f.index.to_series().diff().median() / pd.Timedelta("1h"),
        "средняя ставка за выплату, %": f.mean() * 100,
        "средний funding за сутки, %": fund_daily[a].mean() * 100,
        "средний funding, % годовых": fund_daily[a].mean() * 365 * 100,
        "доля положительных выплат": (f > 0).mean(),
        "мин. ставка за выплату, %": f.min() * 100,
        "макс. ставка за выплату, %": f.max() * 100,
    } for a, f in [("sol", fund_sol), ("btc", fund_btc)]
})
fund_table

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for a in ["sol", "btc"]:
    axes[0].plot(fund_daily[a].rolling(30).mean() * 365 * 100, color=COLORS[a], label=a.upper(), lw=1)
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_title("Funding, % годовых (скользящее среднее за 30 дней)")
axes[0].legend()

axes[1].plot(np.log(sol["close"]), color=COLORS["sol"], lw=1)
axes[1].set_title("Лог-цена фьючерса SOL — для сопоставления с фазами funding")
plt.tight_layout()
plt.show()

In [ ]:
# Связан ли сегодняшний funding с будущей доходностью SOL?
# Будущая лог-доходность за h дней: ln(P_{t+h} / P_t). Funding дня t известен к закрытию дня t (утечки нет).
logp = np.log(sol["close"])
HORIZONS = [1, 3, 7]
fwd = pd.DataFrame({f"{h}д": logp.shift(-h) - logp for h in HORIZONS})
df_fp = fwd.join(fund_daily["sol"].rename("funding")).dropna()

pred_rows = {}
for h in HORIZONS:
    col = f"{h}д"
    rho, p = stats.spearmanr(df_fp["funding"], df_fp[col])
    pred_rows[f"горизонт {col}"] = {"корреляция Спирмена": rho, "p-value (без поправки)": p, "наблюдений": len(df_fp)}
pred_table = pd.DataFrame(pred_rows).T
print("Внимание: для горизонтов 3 и 7 дней окна перекрываются, поэтому p-value занижено.")
pred_table

In [ ]:
# Пятёрки по уровню funding: что было с ценой SOL в следующие 7 дней
df_fp["квинтиль funding"] = pd.qcut(df_fp["funding"], 5, labels=["1 (самый низкий)", "2", "3", "4", "5 (самый высокий)"])
quint = df_fp.groupby("квинтиль funding", observed=True).agg(
    funding_за_сутки_pct=("funding", lambda x: x.mean() * 100),
    доходность_7д_средняя=("7д", "mean"),
    доходность_7д_медиана=("7д", "median"),
    доля_роста_7д=("7д", lambda x: (x > 0).mean()),
    дней=("7д", "size"),
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(5), quint["доходность_7д_медиана"] * 100, color=COLORS["sol"])
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(5), quint.index)
ax.set_ylabel("медианная доходность за 7 дней, %")
ax.set_title("Доходность SOL через 7 дней по уровню funding")
plt.tight_layout()
plt.show()
quint

In [ ]:
# Во что обходится удержание позиции из-за funding (без учёта комиссий биржи)
cost = pd.DataFrame({
    a.upper(): {
        "лонг, % годовых (минус = платим)": -fund_daily[a].mean() * 365 * 100,
        "шорт, % годовых (плюс = получаем)": fund_daily[a].mean() * 365 * 100,
        "худший месяц для лонга, %": -fund_daily[a].resample("ME").sum().max() * 100,
    } for a in ["sol", "btc"]
})
cost

_Вывод будет дописан по результатам запуска на реальных данных._

## 10. Анализ по подпериодам (проверка устойчивости)

Проверяем, держатся ли выводы, если взять не всю историю, а только последние 1.5 года. Если свойства ряда заметно меняются, это признак **смены рыночного режима**: модель нужно проверять на нескольких периодах, а не на одном случайном отрезке.

In [ ]:
RECENT_START = ret.index.max() - pd.DateOffset(months=18)


def period_stats(r: pd.DataFrame, price: pd.Series) -> pd.Series:
    x, b = r["sol"], r["btc"]
    sigma = x.std()
    return pd.Series({
        "наблюдений": len(r),
        "годовая волатильность SOL": sigma * np.sqrt(PERIODS_PER_YEAR),
        "годовая волатильность BTC": b.std() * np.sqrt(PERIODS_PER_YEAR),
        "эксцесс SOL": stats.kurtosis(x),
        "худший день SOL": x.min(),
        "лучший день SOL": x.max(),
        "доля растущих дней SOL": (x > 0).mean(),
        "корреляция SOL–BTC": x.corr(b),
        "бета SOL к BTC": x.cov(b) / b.var(),
        "ACF SOL, лаг 1": acf(x, 1)[0],
        "граница значимости ±": 1.96 / np.sqrt(len(r)),
        "Льюнг–Бокс r, 10 лагов (p)": ljung_box_pvalue(x, 10),
        "Льюнг–Бокс r², 10 лагов (p)": ljung_box_pvalue(x ** 2, 10),
        "corr(SOL_t+1, BTC_t)": x.shift(-1).corr(b),
        "макс. просадка SOL": (price / price.cummax() - 1).min(),
    })


recent = ret[ret.index >= RECENT_START]
subperiods = pd.DataFrame({
    f"весь период ({ret.index.min():%Y-%m-%d} — {ret.index.max():%Y-%m-%d})": period_stats(ret, sol["close"]),
    f"последние 1.5 года ({recent.index.min():%Y-%m-%d} — {recent.index.max():%Y-%m-%d})":
        period_stats(recent, sol.loc[sol.index >= RECENT_START, "close"]),
})
subperiods

**Вывод.** Часть выводов устойчива, часть нет.

- **Устойчиво:** направление по-прежнему как монетка (≈ 49% растущих дней), волатильность по-прежнему кластеризуется (Льюнг–Бокс для r², p ≈ 0), вчерашний BTC по-прежнему не предсказывает SOL (0.006).
- **Изменилось:** за последние 1.5 года SOL стала заметно спокойнее (волатильность **67% против 113%**, эксцесс 2.0 против 8.5, худший день −16% против −55%) и гораздо сильнее привязана к BTC (корреляция **0.83 против 0.59**, бета 1.37 против 1.17).
- **Ослабло:** автокорреляция доходностей на лаге 1 сменила знак и стала незначимой (+0.048 при границе ±0.084): откат после сильного дня, видимый на всём периоде, — скорее свойство ранних лет. Совместный тест Льюнга–Бокса по 10 лагам всё ещё на грани значимости (p = 0.036), то есть слабая память в доходностях SOL есть, но на конкретный лаг опираться нельзя.

**Следствие для модели:** параметры рынка нестабильны во времени, поэтому валидация — только walk-forward, с проверкой качества отдельно на последнем периоде. На коротком отрезке данных в 4 раза меньше и нет кризисов, поэтому обучать модель стоит на всей истории; вариант — давать недавним данным больший вес.

## 11. Итоги и выводы для моделирования

In [ ]:
summary = pd.Series({
    "период": f"{ret.index.min():%Y-%m-%d} — {ret.index.max():%Y-%m-%d}",
    "наблюдений": len(ret),
    "годовая волатильность SOL": f"{ret_stats.loc['годовая волатильность', 'SOL']:.0%}",
    "годовая волатильность BTC": f"{ret_stats.loc['годовая волатильность', 'BTC']:.0%}",
    "эксцесс доходностей SOL": f"{ret_stats.loc['эксцесс (excess)', 'SOL']:.1f}",
    "p-value Жарка–Бера (SOL)": f"{ret_stats.loc['Жарк–Бера, p-value', 'SOL']:.2g}",
    "ADF p-value: log-цена SOL": f"{adf_table.loc['log-цена SOL', 'p-value']:.3f}",
    "ADF p-value: доходность SOL": f"{adf_table.loc['доходность SOL', 'p-value']:.2g}",
    "корреляция SOL–BTC": f"{pearson:.2f}",
    "бета SOL к BTC": f"{beta:.2f}",
    "corr(SOL_t+1, BTC_t)": f"{lead.iloc[0, 0]:.3f}",
    "corr(SOL_t+1, SOL_t)": f"{lead.iloc[0, 1]:.3f}",
    "корреляция доходностей фьючерс–спот (SOL)": f"{tracking.loc['корреляция доходностей фьючерс–спот', 'SOL']:.4f}",
    "средний базис SOL": f"{tracking.loc['средний базис', 'SOL']:.3%}",
    "средний funding SOL, % годовых": f"{fund_table.loc['средний funding, % годовых', 'SOL']:.1f}",
    "доля положительных выплат funding (SOL)": f"{fund_table.loc['доля положительных выплат', 'SOL']:.0%}",
    "Спирмен: funding_t и доходность SOL за 7 дней": f"{pred_table.loc['горизонт 7д', 'корреляция Спирмена']:.3f}",
})
summary.to_frame("значение")

In [ ]:
# Сохраняем лог-доходности для следующих этапов
PROCESSED.mkdir(parents=True, exist_ok=True)
ret.to_csv(PROCESSED / f"log_returns_{INTERVAL}.csv")
fund_daily.to_csv(PROCESSED / "funding_daily.csv")
print("Сохранено:", PROCESSED / f"log_returns_{INTERVAL}.csv", "и", PROCESSED / "funding_daily.csv")

### Основные выводы

1. **Данные.** 2270 дневных свечей SOL/USDT и BTC/USDT с Binance (11.08.2020 — 28.10.2026), без пропусков и ошибок.
2. **Цель — доходность, а не цена.** Лог-цена нестационарна (ADF p = 0.15), лог-доходность стационарна (p ≈ 0).
3. **Высокий риск и тяжёлые хвосты.** Годовая волатильность SOL 113% (у BTC 57%), эксцесс 8.5, экстремальные дни в ~5 раз чаще нормального, максимальная просадка −96%. Нужны устойчивые к выбросам метрики.
4. **Направление предсказывать трудно.** Доля растущих дней ≈ 50%, автокорреляция доходностей SOL слабая (лаг 1: −0.046), у BTC её нет вовсе. Обязательно сравнение с наивными бейзлайнами.
5. **Волатильность предсказуема.** Сильная кластеризация волатильности (Льюнг–Бокс для r², p ≈ 0), эффект выходных (волатильность ниже на ~20–40%).
6. **Связь с BTC сильная, растущая, но без опережения.** Корреляция 0.59 за весь период (0.29 в 2020 → 0.88 в 2026), бета 1.17; вчерашний BTC не предсказывает завтрашний SOL.
7. **Смена режимов.** Волатильность SOL снижается год от года (169% → 62%), связь с BTC растёт. Модель надо валидировать на нескольких периодах, признаки — нормировать.
8. **Проверка устойчивости.** За последние 1.5 года волатильность SOL 67% (против 113% за всё время), корреляция с BTC 0.83 (против 0.59); непредсказуемость направления и кластеризация волатильности сохраняются. Параметры нестабильны во времени — довод за walk-forward валидацию.

### План для чекпойнта 3

- **Целевая переменная:** лог-доходность SOL на горизонте $h$ (кандидаты: 1, 3 и 7 дней). Параллельно — бинарная задача «вырастет / упадёт».
- **Признаки:** лаги доходностей SOL и BTC, скользящая волатильность и размах high–low, объём и его изменение, скользящая корреляция и бета к BTC, признак выходного дня.
- **Валидация:** только по времени, без перемешивания (walk-forward / `TimeSeriesSplit`), признаки строятся строго из прошлого. Отдельно проверить качество без 2020 года.
- **Метрики:** MAE, RMSE, точность направления; бейзлайны — нулевая доходность, «завтра как сегодня», buy & hold.